<a href="https://colab.research.google.com/github/Kanishaag/AI-Fashion-Recommendation-Engine/blob/main/Fashion_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*AI Fashion Recommendation Engine using NLP, Transformers, and Semantic Search*

1. Installing and Importing Dataset Library


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

2. Loading the Research Paper Dataset

In [ ]:
dataset = load_dataset("Censius-AI/ECommerce-Women-Clothing-Reviews", split = "train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print(dataset)

Dataset({
    features: ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name'],
    num_rows: 23486
})


In [ ]:
dataset[0]

{'Unnamed: 0': 0,
 'Clothing ID': 767,
 'Age': 33,
 'Title': None,
 'Review Text': 'Absolutely wonderful - silky and sexy and comfortable',
 'Rating': 4,
 'Recommended IND': 1,
 'Positive Feedback Count': 0,
 'Division Name': 'Initmates',
 'Department Name': 'Intimate',
 'Class Name': 'Intimates'}

3. Converting Dataset into Pandas DataFrame

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(dataset)
df

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...,...
23481,23481,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23482,23482,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23483,23483,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23484,23484,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


In [ ]:
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

4. Selecting Required Columns

In [ ]:
df = df[["Title", "Review Text"]].copy()

In [ ]:
df.shape

(23486, 2)

5. Checking Missing Values

In [ ]:
df.isnull().sum()

,0
Title,3810
Review Text,845


6. Creating & Displaying  Combined Text Column

In [ ]:
df["paper_text"] = df["Title"] + " " + df["Review Text"]

In [ ]:
df[["paper_text"]].head()

,paper_text
0,NaN
1,NaN
2,Some major design flaws I had such high hopes ...
3,"My favorite buy! I love, love, love this jumps..."
4,Flattering shirt This shirt is very flattering...


In [ ]:
print(df["paper_text"].iloc[0])

nan


7. Reducing Dataset Size

In [ ]:
df = df.dropna(subset=["paper_text"])

In [ ]:
df = df.head(10000)
df

,Title,Review Text,paper_text
2,Some major design flaws,I had such high hopes for this dress and reall...,Some major design flaws I had such high hopes ...
3,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...","My favorite buy! I love, love, love this jumps..."
4,Flattering shirt,This shirt is very flattering to all due to th...,Flattering shirt This shirt is very flattering...
5,Not for the very petite,"I love tracy reese dresses, but this one is no...",Not for the very petite I love tracy reese dre...
6,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,Cagrcoal shimmer fun I aded this in my basket ...
...,...,...,...
11958,So flattering!,I love the way this dress fits and flares in a...,So flattering! I love the way this dress fits ...
11959,Love the stripes!,I love this cut of shirt. the stripes are fun....,Love the stripes! I love this cut of shirt. th...
11960,Striped version adorable,"Contrary to what previous reviewers have said,...",Striped version adorable Contrary to what prev...
11961,Cute and springy,This is a really cute shirt for spring. i love...,Cute and springy This is a really cute shirt f...


8. Loading Sentence Transformer Model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [ ]:
print(type(df["paper_text"].iloc[0]))

<class 'str'>


9. Cleaning Text

In [ ]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex = False)
df["paper_text"] = df["paper_text"].str.strip()

In [ ]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Some major design flaws I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c'

10. Generating Embeddings

In [ ]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [ ]:
embedding[:64]

array([ 0.01269737,  0.11324068,  0.09964056,  0.08746878,  0.08028307,
       -0.0659336 , -0.02101759,  0.03151058, -0.03906323,  0.07004053,
       -0.05253115,  0.04169858,  0.00293736, -0.10479511, -0.07227391,
       -0.01814312,  0.07030395, -0.00575975, -0.00706785, -0.02348911,
       -0.10304376, -0.06236155,  0.00321191,  0.04346538, -0.00240702,
       -0.07976666, -0.04643388,  0.06325215, -0.01936218, -0.11576267,
       -0.09096038,  0.06463841,  0.03546961,  0.03819012, -0.01620268,
       -0.00136719,  0.07951716, -0.00825935, -0.03598492,  0.01680012,
        0.00861815, -0.00832039, -0.07782343,  0.06724366, -0.03927702,
       -0.01871645,  0.03669625,  0.01883383, -0.07812978,  0.00367536,
       -0.0099721 ,  0.01284315, -0.04754497,  0.00995371, -0.02223879,
        0.05556561, -0.09300767,  0.00465963, -0.00828252, -0.02883783,
        0.02568153,  0.03207554, -0.04741186,  0.06974019], dtype=float32)

11. Creating Embeddings for Multiple Reviews

In [ ]:
sample_embedding = model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

(5, 384)


12. Calculating Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1),
                            sample_embedding[0].reshape(1, -1))
print(similarity)

[[0.9999999]]


In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1),
                               sample_embedding[1].reshape(1, -1))
print(similarity)

[[0.37833738]]


In [ ]:
for i in  range(1, 5):
  sim = cosine_similarity(sample_embedding[0].reshape(1, -1),
                          sample_embedding[i].reshape(1, -1))
  print(sim)

[[0.37833738]]
[[0.37890348]]
[[0.5945415]]
[[0.51409364]]


13. Saving Embeddings

In [ ]:
import os
import numpy as np

if os.path.exists("fashion_embeddings.npy"):
    print("Loading saved fashion embeddings...")
    embeddings = np.load("fashion_embeddings.npy")
else:
    print("Generating brand-new fashion embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("fashion_embeddings.npy", embeddings)
    print("Fashion embeddings saved successfully!")

Loading saved fashion embeddings...


In [ ]:
print(embeddings.shape)

(10000, 384)


In [ ]:
print(type(embedding))

<class 'numpy.ndarray'>


In [ ]:
embedding.dtype

dtype('float32')

14. Installing and Importing FAISS

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

15. Normalizing Embeddings

In [ ]:
faiss.normalize_L2(embeddings)

16. Creating FAISS Index

In [ ]:
index = faiss.IndexFlatIP(384)

17. Adding Embeddings to Index

In [ ]:
index.add(embeddings)

In [ ]:
if os.path.exists("fashion_embeddings.index"):
    print("Loading existing Fashion FAISS index")
    index = faiss.read_index("fashion_embeddings.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "fashion_embeddings.index")
    print("Fashion FAISS index saved successfully!")

Loading existing Fashion FAISS index


In [ ]:
print(f"Total fashion items indexed: {index.ntotal}")

Total fashion items indexed: 10000


In [ ]:
query = "lightweight and breathable dress for hot summer beach days"

In [ ]:
query_embedding = model.encode([query])
print(query_embedding.shape)

(1, 384)


18. Creating the Search Function


In [ ]:
def search_fashion_item(query, k = 5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    return D, I

19. Searching for Similar Review

In [ ]:
D, I = search_fashion_item(query, 5)
print("Similarity Scores:\n", D)
print("Row Indexes:\n", I)

Similarity Scores:
 [[0.7215053  0.7215053  0.7105434  0.7063663  0.70353866]]
Row Indexes:
 [[9263 8959 5828  519  973]]


In [ ]:
for score, idx in zip(D[0], I[0]):
    print(f"Similarity Score: {score:.2f}")
    print(f"Product Title: {df.iloc[idx]['Title']}")
    print(f"Review Text: {df.iloc[idx]['Review Text']}")
    print("-" * 50 + "\n")

Similarity Score: 0.72
Product Title: Cute cover-up or summer top & shorts!
Review Text: Lightweight, soft cotton top and shorts. i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when i ordered it, but i like it and it matches the look in the photos. and the shorts are very low-cut - don't expect them up around your waist. again, i like that. some might want to wear a cami underneath because it's a thin cotton but i'm fine as-is. i bought it i
--------------------------------------------------

Similarity Score: 0.72
Product Title: Cute cover-up or summer top & shorts!
Review Text: Lightweight, soft cotton top and shorts. i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when i ordered it, but i like it and 

20. Installing Transformers

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline

21. Initializing BART Summarization Pipeline

In [ ]:
summarizer = pipeline(
    "summarization",
    model = "facebook/bart-large-cnn"

)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

22. Generating Summary


In [ ]:
summary = summarizer(df.iloc[642]["Review Text"], max_length = 64, min_length = 32)
print(summary)

[{'summary_text': "i wanted it to work so much but i couldn't get past the itch factor. when i wore a light tee underneath, it didn't move well and looked bulky. if the material doesn't bother you, buy this. would look great w/ tights/leggings and boots."}]


In [ ]:
type(summary)

list

In [ ]:
type(summary[0])

dict

In [ ]:
summary[0]["summary_text"]

"i wanted it to work so much but i couldn't get past the itch factor. when i wore a light tee underneath, it didn't move well and looked bulky. if the material doesn't bother you, buy this. would look great w/ tights/leggings and boots."

23. Creating the Final Search and Summarize Function

In [ ]:
def search_and_summarize_fashion(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    print("="*60)
    print(f"RECOMMENDED OUTFITS FOR: '{query.upper()}'")
    print("="*60 + "\n")

    for score, idx in zip(D[0], I[0]):
        review_title = df.iloc[idx]["Title"]
        full_review = df.iloc[idx]["Review Text"]

        print("Similarity Score:", round(float(score), 2))
        print("Review Title:", review_title)

        summary = summarizer(
            full_review,
            max_length=50,
            min_length=20,
            do_sample=False
        )

        print("AI Summary of Reviews:", summary[0]["summary_text"])
        print("-" * 50 + "\n")

In [ ]:
search_and_summarize_fashion("cute cover-up or summer top & shorts", k=3)

RECOMMENDED OUTFITS FOR: 'CUTE COVER-UP OR SUMMER TOP & SHORTS'

Similarity Score: 0.78
Review Title: Cute cover-up or summer top & shorts!
AI Summary of Reviews: i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when
--------------------------------------------------

Similarity Score: 0.78
Review Title: Cute cover-up or summer top & shorts!


Your max_length is set to 50, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)


AI Summary of Reviews: i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when
--------------------------------------------------

Similarity Score: 0.68
Review Title: Cute summer top
AI Summary of Reviews: This is a nice lightweight summer top to wear with a pair of shorts. It is low cut though - lower than i expected so i wear a tank underneath it.
--------------------------------------------------



24. Installing KeyBERT

In [ ]:
!pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

25. Initializing KeyBERT

In [ ]:
kw_model = KeyBERT()

In [ ]:
type(kw_model)

keybert._model.KeyBERT

26. Keyword Mining

In [ ]:
text = df.iloc[106]["Review Text"]
keywords = kw_model.extract_keywords(text)
print(keywords)

[('skirt', 0.5668), ('styled', 0.4332), ('sweater', 0.3845), ('jacket', 0.3819), ('flattering', 0.3717)]


In [ ]:
print(type(keywords))

<class 'list'>


In [ ]:
print(type(keywords[0]))

<class 'tuple'>


27. End-to-End Search, Summarization and Tagging Pipeline

In [ ]:
def deep_fashion_search(query, k=3):
    # 1. AI Vector Search
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    print("="*60)
    print(f"AI ENGINE RESULTS FOR: '{query.upper()}'")
    print("="*60 + "\n")

    # 2. Extract, Summarize, and Tag Data
    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["Title"]
        review = df.iloc[idx]["Review Text"]

        print(f"Similarity Score: {round(float(score), 2)}")
        print(f"Review Title: {title}")

        # Summary Generation
        summary = summarizer(review, max_length=50, min_length=20, do_sample=False)
        print(f"AI Summary: {summary[0]['summary_text']}")

        # Keyphrase Mining
        keywords = kw_model.extract_keywords(
            review,
            keyphrase_ngram_range=(1, 3),
            stop_words="english",
            top_n=3
        )

        # Format and print the keywords
        extracted_tags = [kw[0] for kw in keywords]
        print(f"Extracted Fashion Tags: {', '.join(extracted_tags)}")
        print("-" * 50 + "\n")

In [ ]:
deep_fashion_search("cute cover-up or summer top & shorts")

AI ENGINE RESULTS FOR: 'CUTE COVER-UP OR SUMMER TOP & SHORTS'

Similarity Score: 0.78
Review Title: Cute cover-up or summer top & shorts!
AI Summary: i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when
Extracted Fashion Tags: lightweight soft cotton, soft cotton shorts, cotton shorts
--------------------------------------------------

Similarity Score: 0.78
Review Title: Cute cover-up or summer top & shorts!
AI Summary: i think it's meant to be a beach cover-up but i'm wearing it as a thin, light-weight summer outfit on these hot hot days. the top has a loose elastic around the bottom which i didn't realize when


Your max_length is set to 50, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)


Extracted Fashion Tags: lightweight soft cotton, soft cotton shorts, cotton shorts
--------------------------------------------------

Similarity Score: 0.68
Review Title: Cute summer top
AI Summary: This is a nice lightweight summer top to wear with a pair of shorts. It is low cut though - lower than i expected so i wear a tank underneath it.
Extracted Fashion Tags: lightweight summer wear, wear tank underneath, tank underneath layers
--------------------------------------------------



28. Implementing Named Entity Recognition (NER) Processes

#Approach 1: Hugging Face Transformers for Feature Extraction (BERT for NER)

In [ ]:
from transformers import pipeline

In [ ]:
hf_ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def run_approach_one_ner(text_to_test):
    print("-" * 50)
    print("OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER")

    results = hf_ner_pipeline(str(text_to_test))

    if not results:
        print("No standard entities found.")
    else:
        for entity in results:
            print(f"Found Entity: '{entity['word']}' ---> Label: {entity['entity_group']} (Confidence: {round(entity['score'], 2)})")

In [ ]:
sample_text_1 = "I wore this amazing sundress on my vacation to Paris and California!"
run_approach_one_ner(sample_text_1)

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
Found Entity: 'Paris' ---> Label: LOC (Confidence: 1.0)
Found Entity: 'California' ---> Label: LOC (Confidence: 1.0)


# Approach 2: Custom Dictionary-Based Fashion-NER

In [ ]:
import re

In [ ]:
def run_approach_two_ner(text_to_test):
    print("-" * 50)
    print("OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER")

    fashion_dictionary = {
        "Fabric": r"\b(cotton|linen|silk|denim|leather|lace|knit|wool|polyester|rayon|blend|satin)\b",
        "Size & Style": r"\b(petite|regular|plus size|oversized|true to size|small|medium|large|xs|xl|xxl|short|long|loose|tight)\b",
        "Categories": r"\b(dress|top|shorts|pants|skirt|blouse|jeans|jacket|sweater|maxi|cover-up|romper|jumpsuit|tee|shirt)\b"
    }

    found_any = False
    for category, regex_pattern in fashion_dictionary.items():
        matches = set(re.findall(regex_pattern, str(text_to_test), flags=re.IGNORECASE))
        if matches:
            found_any = True
            print(f"{category} Detected: {', '.join(matches)}")

    if not found_any:
        print("No specific fashion tags matched the dictionary rules.")

In [ ]:
sample_text_2 = "I bought this lovely cotton and linen blend maxi dress. It is beautifully oversized!"
run_approach_two_ner(sample_text_2)

--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Fabric Detected: blend, linen, cotton
Size & Style Detected: oversized
Categories Detected: dress, maxi


29. Final Engine Pipeline

In [ ]:
def deep_fashion_search_final(query, k=2):
    # FAISS Vector Search matching
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    print(f"COMPLETE AI ENGINE PIPELINE FOR QUERY: '{query.upper()}'")
    print("-" * 60 + "\n")

    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["Title"]
        review = df.iloc[idx]["Review Text"]

        print(f"Similarity Score: {round(float(score), 2)}")
        print(f"Review Title: {title}")

        # Review Summarization using BART
        summary_result = summarizer(str(review), max_length=50, min_length=20, do_sample=False)
        print(f"AI Summary: {summary_result[0]['summary_text']}\n")

        # Run both stable NER approaches
        run_approach_one_ner(str(review))
        run_approach_two_ner(str(review))

        print("=" * 60 + "\n")

30. Testing complete engine

In [ ]:
deep_fashion_search_final("party wear, sparkle and lightweight dress for a hot sunny beach vacation", k=2)

Your max_length is set to 50, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


COMPLETE AI ENGINE PIPELINE FOR QUERY: 'PARTY WEAR, SPARKLE AND LIGHTWEIGHT DRESS FOR A HOT SUNNY BEACH VACATION'
--------------------------------------------------

Similarity Score: 0.62
Review Title: Fun dress


Your max_length is set to 50, but your input_length is only 34. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=17)


AI Summary: An easy dress to wear - good choice for both day and evening. Extremely flattering.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Categories Detected: dress

Similarity Score: 0.6
Review Title: Fun!
AI Summary: This is my go to, cool, summer dress. it's very light and airy and extremely comfortable. great for 90 degree days!highly recommend!

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Categories Detected: dress



In [ ]:
deep_fashion_search_final("cotton shirt for daily wear and office wear", k=5)

Your max_length is set to 50, but your input_length is only 46. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=23)


COMPLETE AI ENGINE PIPELINE FOR QUERY: 'COTTON SHIRT FOR DAILY WEAR AND OFFICE WEAR'
------------------------------------------------------------

Similarity Score: 0.61
Review Title: Great shirt for work


Your max_length is set to 50, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)


AI Summary: This top is really comfortable but dressy enough for a business/casual office. The fit is nice and loose without making you look larger than you are. i get a lot of compliments every time i wear this.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Size & Style Detected: loose
Categories Detected: top

Similarity Score: 0.61
Review Title: Day to night


Your max_length is set to 50, but your input_length is only 15. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)


AI Summary: This shirt is the perfect transition from the office to happy hour. If you're looking for a great day to night shirt this is it.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Categories Detected: shirt

Similarity Score: 0.57
Review Title: Casual
AI Summary: This shirt is the perfect way to dress down for a long day at school. The shirt is available in sizes 8-12.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Size & Style Detected: long
Categories Detected: dress, shirt

Similarity Score: 0.57
Review Title: Somehow it did not work?
AI Summary: I liked its floral pattern and shape of sleeves. texture was ok. I tried

Your max_length is set to 50, but your input_length is only 23. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)


No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Fabric Detected: linen
Categories Detected: shirt

Similarity Score: 0.57
Review Title: Great shirt
AI Summary: "I love an all cotton shirt! feels great and has a bit of quirky styling without being over the top"

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Fabric Detected: cotton
Categories Detected: shirt, top



31. Interactive AI Fashion Search Prompt

In [ ]:
user_query = input("What kind of outfit are you looking for today? ")
deep_fashion_search_final(user_query, k=2)

What kind of outfit are you looking for today? baggy jeans


Your max_length is set to 50, but your input_length is only 21. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=10)


COMPLETE AI ENGINE PIPELINE FOR QUERY: 'BAGGY JEANS'
------------------------------------------------------------

Similarity Score: 0.7
Review Title: Nice jeans


Your max_length is set to 50, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)


AI Summary: Good quality jeans, but had to return, as they did not fit my curvy hips.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Categories Detected: jeans

Similarity Score: 0.7
Review Title: Waist gets baggy with wear
AI Summary: The waist is really baggy. the fabric stretches with wear and takes a wash to get back to its original shape.

--------------------------------------------------
OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER
No standard entities found.
--------------------------------------------------
OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER
Categories Detected: jeans

